# COS30082 · Approach 2 · Cross-crop disease recognition

The previous DINOv2 Baseline 2 run gave 9.60% unseen Top‑1 over 333 candidates and 87.50% within the 10 unseen candidates. On a separate train-derived probe, the crop and disease heads reached 80.94% and 18.75% on proxy-unseen images. This experiment measures whether disease recognition improves across crops and how calibrated stacking changes the ranking of the same predictions.

Baseline 2 is trained in the team's separate baseline notebook. Use a verified DINOv2 run for the official 333-candidate comparison. Baseline 2 also differs in its heads, label smoothing, random augmentation, cosine schedule and image-level validation, so use the controlled Approach 2 ablations to estimate individual component effects.

## 1. Setup

Open this notebook in the team repository and use the VS Code Jupyter kernel with the team requirements. Set `DEBUG=True` to check two batches per recipe with random backbone weights. Set `DEBUG=False` and choose **Run All** for pretrained development, full-train refit, and official evaluation.

In [1]:
DEBUG = False  # True for a quick check. False for the full run.

from pathlib import Path
from itertools import islice
from time import perf_counter
import hashlib
import json
import sys

import numpy as np
import pandas as pd
import timm
import torch
from IPython.display import display
from PIL import Image
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset

here = Path.cwd()
PROJECT_ROOT = next((p for p in (here, *here.parents)
                     if (p / "common" / "dataset.py").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Open this notebook inside the team repository.")
sys.path.insert(0, str(Path(PROJECT_ROOT).resolve()))

from common import config as team
from common.dataset import PlantDataset, get_pairs, get_transforms, load_split
from common.evaluate import PERTURBATIONS, harmonic_mean, save_confusion, top_k_accuracy

BACKBONE = "vit_base_patch14_dinov2.lvd142m"
SEED = 73
EPOCHS = 1 if DEBUG else 20
BATCH_SIZE = 8 if DEBUG else 32
ENCODE_BATCH_SIZE = 8 if DEBUG else 16
PROJECTION_DIM = 128
LR, WEIGHT_DECAY = 1e-4, 1e-4
DECORRELATION_WEIGHT, SUPCON_WEIGHT, TEMPERATURE = 0.05, 0.1, 0.1
MATCHES_PER_BATCH = 4
FLIP_PROBABILITY = 0.5
PAIRS_PER_GROUP, MIN_PROXY_IMAGES = 5, 5
CALIBRATION_GRID = [step / 2 for step in range(25)]
MAX_SEEN_DROP_PP, MIN_PROXY_GAIN_PP = 5.0, 1.0
DEVICE = team.DEVICE
USE_BF16 = DEVICE.type == "cuda" and torch.cuda.is_bf16_supported()
OUTPUT_DIR = team.RESULTS_DIR / ("approach2_final_dinov2" + ("_debug" if DEBUG else ""))
DEVELOPMENT_LOCK = OUTPUT_DIR / "development_lock.json"
FINAL_LOCK = OUTPUT_DIR / "final_lock.json"
CHECKPOINT = OUTPUT_DIR / "final_heads.pt"
OFFICIAL_REPORT = OUTPUT_DIR / "official_report.json"
RECIPES = {
    "A2a": (0.0, 0.0, True),
    "A2b": (DECORRELATION_WEIGHT, 0.0, True),
    "A2c": (DECORRELATION_WEIGHT, SUPCON_WEIGHT, True),
    "A2c_no_flip": (DECORRELATION_WEIGHT, SUPCON_WEIGHT, False),
}

if team.NUM_PAIRS != team.NUM_SEEN_PAIRS + team.NUM_UNSEEN_PAIRS:
    raise RuntimeError("Use the current team common/ files with the 323/333 protocol.")
if team.IMAGE_SIZE % 14:
    raise ValueError("DINOv2 ViT-B/14 requires an image size divisible by 14.")
if EPOCHS < 1 or not 0 < team.VAL_FRACTION < 1 or BATCH_SIZE < 8:
    raise ValueError("Check epochs, validation fraction and batch size.")
print(f"Python {sys.version.split()[0]} | torch {torch.__version__} | {DEVICE} | debug={DEBUG}")

c:\Users\104394862\AppData\Local\COS30082-AI-LAB\venv-py313-cu128\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Python 3.13.5 | torch 2.7.1+cu128 | cuda | debug=False


## 2. Data

All development splits come from `DPD_Small_train.csv`. Hold out ten **complete** crop–disease combinations with seed 73 and at least five images per pair. Retain both concepts of each held-out pair elsewhere in fit. Five held-out pairs are for selection and five are for the held-out development check. For the remaining pairs, sample about 10% of images per pair for seen validation and divide those images into the same two groups. The tables below show the split sizes and how many fit images and crops support each proxy disease. Concept names and IDs are checked before training.

Every development image is ranked over all 323 train-derived candidates. Both proxy groups are held out of fitting. The flip-enabled recipes use a horizontal flip on fit images with probability 0.5. The full-train refit follows the selected recipe. Validation and final test use the team's deterministic ImageNet transform.

In [2]:
def image_key(value):
    result = Path(str(value).strip().replace("\\", "/"))
    if not result.name or result.is_absolute() or ".." in result.parts or ":" in str(result):
        raise ValueError(f"Expected a relative image filename, got {value!r}.")
    return result.as_posix()


def sha256_file(path):
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()


if not team.TRAIN_CSV.is_file() or not team.IMAGE_DIR.is_dir():
    raise FileNotFoundError(f"Expected {team.TRAIN_CSV} and {team.IMAGE_DIR}. Run the team's dataset setup.")

train_full = load_split(team.TRAIN_CSV)
for concept in ("crop", "disease"):
    names = train_full[f"{concept}_name"].astype("string").str.strip().str.casefold()
    if names.isna().any() or names.eq("").any():
        raise ValueError(f"Missing {concept} names in train CSV.")
    mapping = train_full[[concept]].assign(name=names).drop_duplicates()
    if mapping[concept].duplicated().any() or mapping["name"].duplicated().any():
        raise ValueError(f"Inconsistent {concept} name/ID mapping in train CSV.")
train_full["image"] = train_full["image"].map(image_key)
if not train_full.image.is_unique:
    raise ValueError("Repeated training image paths would compromise the split.")
if not (team.IMAGE_DIR / train_full.iloc[0]["image"]).is_file():
    raise FileNotFoundError(f"Image paths are relative to {team.IMAGE_DIR}.")
train_pairs = get_pairs()
if train_pairs.shape != (team.NUM_SEEN_PAIRS, 2):
    raise ValueError("Development must use only training-derived candidate pairs.")
if not torch.equal(train_pairs[train_full.pair_row.to_numpy(copy=True)],
                   torch.as_tensor(train_full[["crop", "disease"]].to_numpy(copy=True))):
    raise ValueError("Training labels do not match the shared candidate order.")
if (train_pairs.min() < 0 or train_pairs[:, 0].max() >= team.NUM_CROPS or
        train_pairs[:, 1].max() >= team.NUM_DISEASES):
    raise ValueError("Crop or disease concept ID exceeds the shared class vocabulary.")

# Train-derived pairs used in the earlier exploratory diagnostic.
PREVIOUS_PROBE = {
    ("basil", "healthy"), ("bean", "rust"), ("cassava", "brown_spot"),
    ("cherry", "healthy"), ("cucumber", "anthracnose"), ("paddy", "downy_mildew"),
    ("raspberry", "healthy"), ("soybean", "frog_eye_leaf_spot"),
    ("tomato", "late_blight"), ("tomato", "spider_mites"),
}


def split_training(frame):
    rng = np.random.default_rng(SEED)
    combos = frame.groupby(["pair_row", "crop", "disease"], sort=True).size().rename("n").reset_index()
    representatives = frame.drop_duplicates("pair_row")
    previously_used = {
        row.pair_row for row in representatives.itertuples()
        if (row.crop_name.strip().lower(), row.disease_name.strip().lower()) in PREVIOUS_PROBE
    }
    eligible = combos[(combos.n >= MIN_PROXY_IMAGES) & ~combos.pair_row.isin(previously_used)]
    required = 2 * PAIRS_PER_GROUP
    for _ in range(128):
        crop_count = combos.groupby("crop").size().to_dict()
        disease_count = combos.groupby("disease").size().to_dict()
        held = []
        for row in eligible.iloc[rng.permutation(len(eligible))].itertuples():
            if crop_count[row.crop] > 1 and disease_count[row.disease] > 1:
                held.append(int(row.pair_row))
                crop_count[row.crop] -= 1
                disease_count[row.disease] -= 1
                if len(held) == required:
                    break
        if len(held) == required:
            break
    else:
        raise ValueError("Too few eligible compositions. Check metadata or adjust holdout settings before development.")

    remaining = frame[~frame.pair_row.isin(held)]
    seen_val = remaining.groupby("pair_row").sample(frac=team.VAL_FRACTION, random_state=SEED)
    seen_val = seen_val.iloc[rng.permutation(len(seen_val))]
    midpoint = len(seen_val) // 2
    frames = {
        "fit": remaining.drop(seen_val.index),
        "tune_seen": seen_val.iloc[:midpoint],
        "tune_proxy": frame[frame.pair_row.isin(held[:PAIRS_PER_GROUP])],
        "check_seen": seen_val.iloc[midpoint:],
        "check_proxy": frame[frame.pair_row.isin(held[PAIRS_PER_GROUP:])],
    }
    if any(part.empty for part in frames.values()):
        raise ValueError("Every development partition needs images.")
    actual = sorted(index for part in frames.values() for index in part.index)
    if actual != frame.index.tolist():
        raise RuntimeError("Development partitions overlap or omit images.")
    fit = frames["fit"]
    if set(fit.pair_row) != set(remaining.pair_row):
        raise ValueError("Seen validation removed an entire fit pair.")
    for part in (frames["tune_proxy"], frames["check_proxy"]):
        if (set(part.pair_row) & set(fit.pair_row) or not set(part.crop) <= set(fit.crop)
                or not set(part.disease) <= set(fit.disease)):
            raise ValueError("Proxy holdout must exclude complete pairs while retaining each concept in fit.")
    return frames


splits = split_training(train_full)
fit_mask = torch.isin(torch.arange(len(train_pairs)),
                      torch.as_tensor(splits["fit"].pair_row.unique()))
_, image_transform = get_transforms("imagenet")
split_summary = pd.DataFrame([
    {"split": name, "images": len(frame), "compositions": frame.pair_row.nunique()}
    for name, frame in splits.items()
])
display(split_summary)

fit_diseases = splits["fit"].groupby("disease").agg(
    fit_disease_images=("image", "size"), fit_disease_crops=("crop", "nunique"))
proxy_coverage = pd.concat([
    splits[name].groupby(["pair_row", "crop_name", "disease", "disease_name"]).size()
    .rename("proxy_images").reset_index().assign(split=name)
    for name in ("tune_proxy", "check_proxy")
]).join(fit_diseases, on="disease")
display(proxy_coverage)
print(f"Fit diseases spanning at least two crops: {fit_diseases.fit_disease_crops.ge(2).sum()}"
      f"/{len(fit_diseases)}")
settings = {
    "backbone": BACKBONE, "seed": SEED, "epochs": EPOCHS, "batch_size": BATCH_SIZE,
    "encode_batch_size": ENCODE_BATCH_SIZE, "image_size": team.IMAGE_SIZE,
    "resize_size": team.RESIZE_SIZE, "val_fraction": team.VAL_FRACTION,
    "projection_dim": PROJECTION_DIM, "lr": LR, "weight_decay": WEIGHT_DECAY,
    "recipes": {key: list(weights) for key, weights in RECIPES.items()},
    "temperature": TEMPERATURE, "matched_pairs": MATCHES_PER_BATCH, "flip_probability": FLIP_PROBABILITY,
    "calibration_grid": CALIBRATION_GRID, "max_seen_drop_pp": MAX_SEEN_DROP_PP,
    "min_proxy_gain_pp": MIN_PROXY_GAIN_PP,
    "selection_rule": "raw tune harmonic mean, then proxy Top-1, seen Top-1, earlier epoch",
    "train_csv_sha256": sha256_file(team.TRAIN_CSV), "device": str(DEVICE),
    "torch": str(torch.__version__), "timm": timm.__version__, "bf16_features": USE_BF16,
    "debug": DEBUG, "pairs_per_group": PAIRS_PER_GROUP, "min_proxy_images": MIN_PROXY_IMAGES,
    "tune_pair_rows": sorted(splits["tune_proxy"].pair_row.unique().tolist()),
    "check_pair_rows": sorted(splits["check_proxy"].pair_row.unique().tolist()),
}

,split,images,compositions
0,fit,7071,313
1,tune_seen,380,134
2,tune_proxy,37,5
3,check_seen,380,131
4,check_proxy,132,5


,pair_row,crop_name,disease,disease_name,proxy_images,split,fit_disease_images,fit_disease_crops
0,36,bean,9,healthy,10,tune_proxy,1415,35
1,78,cauliflower,3,black_rot,7,tune_proxy,27,4
2,135,garlic,10,rust,5,tune_proxy,258,10
3,212,peach,12,monillia_laxa,8,tune_proxy,10,2
4,255,soybean,10,rust,7,tune_proxy,258,10
0,24,banana,9,healthy,11,check_proxy,1415,35
1,211,peach,57,curl,12,check_proxy,33,4
2,234,potato,139,late_blight,70,check_proxy,101,1
3,265,sugarcane,51,mosaic,18,check_proxy,492,2
4,271,sugarcane,58,yellowing,21,check_proxy,18,2


Fit diseases spanning at least two crops: 41/175


## 3. Model

One frozen ImageNet-normalized DINOv2 ViT-B/14 maps each image to its CLS feature. Two learned 128-dimensional projections feed separate crop and disease classifiers. The pair score is

$$s(x,c,d)=\log p(c\mid x)+\log p(d\mid x).$$

Decorrelation penalizes squared batch cross-correlation between the projected branches. This is a soft feature disentanglement objective. Cross-crop supervised contrastive learning pulls together disease embeddings whose disease labels match across different crop species. Other diseases supply negatives. Same-disease, same-crop examples are excluded from the contrastive denominator. Training logs show how many images have a usable positive and negative in each batch.

$$L=\mathrm{CE}_{crop}+\mathrm{CE}_{disease}+\lambda_{dec}L_{dec}+\lambda_{sc}L_{sc}.$$

| Recipe | Training objective | Fit flip |
|---|---|---|
| A2a | Projected crop and disease branches with their two CE losses | Yes |
| A2b | A2a plus decorrelation, weight 0.05 | Yes |
| A2c | A2b plus cross-crop SupCon, weight 0.10 and temperature 0.10 | Yes |
| A2c_no_flip | A2c with the original image features only | No |

In [3]:
class ProjectedHeads(nn.Module):
    def __init__(self, width):
        super().__init__()
        self.crop_projection = nn.Sequential(nn.LayerNorm(width), nn.Linear(width, PROJECTION_DIM), nn.GELU())
        self.disease_projection = nn.Sequential(nn.LayerNorm(width), nn.Linear(width, PROJECTION_DIM), nn.GELU())
        self.crop_head = nn.Linear(PROJECTION_DIM, team.NUM_CROPS)
        self.disease_head = nn.Linear(PROJECTION_DIM, team.NUM_DISEASES)

    def forward(self, features):
        crop = self.crop_projection(features)
        disease = self.disease_projection(features)
        return self.crop_head(crop), self.disease_head(disease), crop, disease


def decorrelation(crop_features, disease_features):
    def standardize(x):
        return (x - x.mean(0)) / (x.var(0, unbiased=False) + 1e-4).sqrt()

    crop_z, disease_z = standardize(crop_features), standardize(disease_features)
    cross_correlation = crop_z.T @ disease_z / len(crop_z)
    return cross_correlation.square().mean()


def cross_crop_supcon(features, crops, diseases):
    z = F.normalize(features, dim=1)
    same_disease = diseases[:, None].eq(diseases)
    positive = same_disease & crops[:, None].ne(crops)
    negative = ~same_disease
    similarities = (z @ z.T) / TEMPERATURE
    log_denominator = similarities.masked_fill(~(positive | negative), -1e9).logsumexp(dim=1)
    counts = positive.sum(1)
    avg_positive = similarities.masked_fill(~positive, 0).sum(1) / counts.clamp_min(1)
    anchors = (counts > 0) & negative.any(1)
    loss = (log_denominator - avg_positive).masked_fill(~anchors, 0.0)
    return loss.sum() / anchors.sum().clamp_min(1), anchors.sum()


def pair_scores(crop_logits, disease_logits, candidates):
    candidates = candidates.to(crop_logits.device)
    crop_scores = F.log_softmax(crop_logits.float(), dim=1)
    disease_scores = F.log_softmax(disease_logits.float(), dim=1)
    return crop_scores[:, candidates[:, 0]] + disease_scores[:, candidates[:, 1]]


def calibrate_scores(scores, seen_membership, gamma):
    return scores - gamma * seen_membership.to(device=scores.device, dtype=scores.dtype)


def create_backbone():
    print("Loading DINOv2 ViT-B/14" + (" (random weights for debug check)" if DEBUG else " pretrained weights"),
          flush=True)
    model = timm.create_model(BACKBONE, pretrained=not DEBUG, num_classes=0, img_size=team.IMAGE_SIZE,
                              pretrained_cfg_overlay={"hf_hub_id": None})
    return model.eval().requires_grad_(False).to(DEVICE)


backbone = None
backbone_record = None


def get_backbone():
    global backbone, backbone_record
    if backbone is None:
        team.set_seed(SEED)
        model = create_backbone()
        digest = hashlib.sha256()
        for name, tensor in sorted(model.state_dict().items()):
            values = tensor.detach().cpu().contiguous().numpy()
            digest.update(f"{name} {values.dtype} {values.shape}\n".encode("utf-8"))
            digest.update(values.tobytes())
        backbone_record = {
            "model": BACKBONE, "timm": timm.__version__,
            "source": "random initialization" if DEBUG else model.pretrained_cfg["url"],
            "state_sha256": digest.hexdigest(),
        }
        backbone = model
    return backbone


def encode_images(images):
    with torch.no_grad(), torch.autocast(DEVICE.type, dtype=torch.bfloat16, enabled=USE_BF16):
        return get_backbone()(images.to(DEVICE)).float()

## 4. Training

Cache the frozen backbone features once per stage. All four recipes have identical initial head weights, minibatch membership and a maximum of 20 development epochs. Each batch tries to contain up to four cross-crop positive pairs. A separate random stream controls flipping.

After every epoch, rank all 323 candidates by the raw score. On the tune group, choose the recipe and epoch by seen/proxy-unseen Top-1 harmonic mean, then proxy-unseen Top-1, seen Top-1 and earlier epoch. Measure all four recipes on the separate check group at that same selected epoch. Compare A2b with A2a for decorrelation, A2c with A2b for SupCon, and A2c with A2c_no_flip for augmentation. The check group estimates the ablation differences.

### Calibrated stacking

After choosing the raw checkpoint, use the tune group to select $\gamma$ on a fixed grid from 0 to 12. Score a candidate as $s(x,c,d)-\gamma\,\mathbf{1}[(c,d)\in S_{fit}]$, where $S_{fit}$ is the vocabulary of combinations represented in fit. For final evaluation the full train CSV defines membership.

Prefer candidates with at least a 1-point gain in proxy-unseen Top-1, a higher harmonic mean, and at most a 5-point seen Top-1 drop. Among those values choose the one with the highest harmonic mean. If none meets those targets, take the best positive value by harmonic mean for the exploratory comparison. Record the same threshold checks on the separate development check. The check group supplies evidence about whether the penalty helped, and both report rows are produced regardless of the outcome.

In [4]:
def atomic_json(path, value):
    temporary = path.with_suffix(".json.tmp")
    temporary.write_text(json.dumps(value, indent=2), encoding="utf-8")
    temporary.replace(path)


def cache_features(frame, name, with_flip=False):
    if DEBUG and len(frame) > 128:
        frame = frame.sample(128, random_state=SEED)
    frame = frame.reset_index(drop=True)
    loader = DataLoader(PlantDataset(frame, image_transform, "imagenet"),
                        batch_size=ENCODE_BATCH_SIZE, shuffle=False, num_workers=0)
    features, flipped, targets = [], [], []
    start = perf_counter()
    for step, (images, crops, diseases, pairs) in enumerate(loader, 1):
        features.append(encode_images(images))
        if with_flip:
            flipped.append(encode_images(images.flip(-1)))
        targets.append(torch.stack((crops, diseases, pairs), dim=1))
        if step == 1 or step == len(loader) or step % 50 == 0:
            print(f"{name}: encoding batch {step}/{len(loader)} | {perf_counter() - start:.1f}s", flush=True)
    return (torch.cat(features), torch.cat(flipped) if flipped else None,
            torch.cat(targets).to(DEVICE))


def batch_indices(labels, rng):
    """Use each image once and include cross-crop positives where available."""
    frame = pd.DataFrame(labels[:, :2].cpu().numpy(), columns=["crop", "disease"])
    groups = []
    for _, disease in frame.groupby("disease"):
        pools = [rng.permutation(crop.index).tolist() for _, crop in disease.groupby("crop")]
        if len(pools) > 1:
            groups.append(pools)
    available = np.ones(len(labels), dtype=bool)
    random_order = iter(rng.permutation(len(labels)))
    for offset in range(0, len(labels), BATCH_SIZE):
        size = min(BATCH_SIZE, len(labels) - offset)
        batch = []
        for _ in range(min(MATCHES_PER_BATCH, size // 2)):
            for group in rng.permutation(len(groups)):
                viable = []
                for pool in groups[group]:
                    while pool and not available[pool[-1]]:
                        pool.pop()
                    if pool:
                        viable.append(pool)
                if len(viable) >= 2:
                    a, b = rng.choice(len(viable), size=2, replace=False)
                    chosen = [viable[a].pop(), viable[b].pop()]
                    available[chosen] = False
                    batch.extend(chosen)
                    break
            else:
                break
        while len(batch) < size:
            row = int(next(random_order))
            if available[row]:
                available[row] = False
                batch.append(row)
        yield torch.as_tensor(batch, device=DEVICE)


def fit_heads(data, recipe, stage, epochs, tune=None):
    team.set_seed(SEED)
    features, flipped, labels = data
    heads = ProjectedHeads(features.shape[1]).to(DEVICE)
    optimizer = torch.optim.AdamW(heads.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    rng = np.random.default_rng(SEED)
    flip_rng = np.random.default_rng(SEED + 1)
    decorrelation_weight, supcon_weight, use_flip = RECIPES[recipe]
    if use_flip and flipped is None:
        raise ValueError("Training recipe requires flipped features.")
    history = []
    best_metrics, epoch_weights = None, {}
    for epoch in range(1, epochs + 1):
        heads.train()
        totals = torch.zeros(5, device=DEVICE)
        examples = 0
        batches = batch_indices(labels, rng)
        if DEBUG:
            batches = islice(batches, 2)
        for idx in batches:
            x = features[idx]
            crop, disease = labels[idx, :2].T
            if use_flip:
                alternate = torch.as_tensor(flip_rng.random(len(idx)) < FLIP_PROBABILITY, device=DEVICE)
                x = torch.where(alternate[:, None], flipped[idx], x)
            crop_logits, disease_logits, crop_z, disease_z = heads(x)
            ce = F.cross_entropy(crop_logits, crop) + F.cross_entropy(disease_logits, disease)
            # Measure usable anchors for every recipe on the same batches.
            con, anchors = cross_crop_supcon(disease_z, crop, disease)
            decor = decorrelation(crop_z, disease_z)
            loss = ce + decorrelation_weight * decor + supcon_weight * con
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()
            totals += torch.stack((loss, ce, decor, con, anchors / len(idx))).detach() * len(idx)
            examples += len(idx)
        values = (totals / examples).tolist()
        if not np.isfinite(values).all():
            raise FloatingPointError(f"Training produced non-finite losses for {recipe}.")
        row = dict(zip(("loss", "ce", "decorrelation", "supcon", "anchor_fraction"), values))
        row = {"stage": stage, "recipe": recipe, "epoch": epoch, **row}
        message = (f"{stage} {recipe} epoch {epoch}/{epochs} | loss {row['loss']:.3f}"
                   f" | usable SupCon anchors {100 * row['anchor_fraction']:.1f}%")
        if tune is not None:
            heads.eval()
            scores = [score_cache(heads, group) for group in tune]
            metrics = {"epoch": epoch, **dev_metrics(*scores)}
            row.update({f"tune_{key}": value for key, value in metrics.items() if key != "epoch"})
            epoch_weights[epoch] = {name: tensor.detach().cpu().clone()
                                    for name, tensor in heads.state_dict().items()}
            if best_metrics is None or selection_key(metrics) > selection_key(best_metrics):
                best_metrics = metrics
            message += (f" | tune seen {metrics['seen_top1']:.2f}%"
                        f" | proxy {metrics['proxy_top1']:.2f}% | H {metrics['harmonic_top1']:.2f}%")
        history.append(row)
        print(message, flush=True)
    if supcon_weight > 0 and not DEBUG and not any(row["anchor_fraction"] > 0 for row in history):
        raise RuntimeError(f"{recipe} had no usable cross-crop positives. Check the fit split.")
    return heads.eval(), history, best_metrics, epoch_weights


@torch.inference_mode()
def score_cache(heads, data):
    features, _, labels = data
    scores, crop_pred, disease_pred = [], [], []
    for batch in features.split(BATCH_SIZE):
        c, d, _, _ = heads(batch)
        scores.append(pair_scores(c, d, train_pairs).cpu())
        crop_pred.append(c.argmax(1).cpu())
        disease_pred.append(d.topk(5, dim=1).indices.cpu())
    return torch.cat(scores), labels.cpu(), torch.cat(crop_pred), torch.cat(disease_pred)


def selection_key(metrics):
    return (metrics["harmonic_top1"], metrics["proxy_top1"],
            metrics["seen_top1"], -metrics["epoch"])


def proxy_details(scored_proxy, gamma=0.0):
    scores, labels, crop, disease_top5 = scored_proxy
    targets = labels[:, 2]
    adjusted = calibrate_scores(scores, fit_mask, gamma)
    predicted = adjusted.topk(5, dim=1).indices
    within_proxy = scores.masked_fill(fit_mask, -torch.inf).topk(min(5, int((~fit_mask).sum())), dim=1).indices
    pair_only_top1 = within_proxy[:, 0].eq(targets)
    true_score = adjusted.gather(1, targets[:, None])
    fields = {
        "pair_row": targets.numpy(),
        "joint_top1": predicted[:, 0].eq(targets).numpy(),
        "joint_top5": predicted.eq(targets[:, None]).any(1).numpy(),
        "proxy_only_top1": pair_only_top1.numpy(),
        "proxy_only_top5": within_proxy.eq(targets[:, None]).any(1).numpy(),
        "crop_top1": crop.eq(labels[:, 0]).numpy(),
        "disease_top1": disease_top5[:, 0].eq(labels[:, 1]).numpy(),
        "disease_top5": disease_top5.eq(labels[:, 1, None]).any(1).numpy(),
        "predicted_seen": fit_mask[predicted[:, 0]].numpy(),
        "rank_blocked": (pair_only_top1 & fit_mask[predicted[:, 0]]).numpy(),
        "correct_pair_rank": (1 + adjusted.gt(true_score).sum(1)).numpy(),
    }
    return pd.DataFrame(fields)


def dev_metrics(scored_seen, scored_proxy, *, gamma=0.0, details=None):
    scores, labels, _, _ = scored_seen
    scores = calibrate_scores(scores, fit_mask, gamma)
    seen_top1 = top_k_accuracy(scores, labels[:, 2], 1)
    details = proxy_details(scored_proxy, gamma) if details is None else details
    proxy_top1 = round(100 * details.joint_top1.mean(), 2)
    return {
        "seen_top1": seen_top1, "proxy_top1": proxy_top1,
        "seen_top5": top_k_accuracy(scores, labels[:, 2], 5),
        "proxy_top5": round(100 * details.joint_top5.mean(), 2),
        "harmonic_top1": harmonic_mean(seen_top1, proxy_top1),
        "proxy_crop_head_top1": round(100 * details.crop_top1.mean(), 2),
        "proxy_disease_head_top1": round(100 * details.disease_top1.mean(), 2),
        "proxy_disease_head_top5": round(100 * details.disease_top5.mean(), 2),
        "proxy_only_top1": round(100 * details.proxy_only_top1.mean(), 2),
        "proxy_only_top5": round(100 * details.proxy_only_top5.mean(), 2),
        "proxy_macro_top1": round(100 * details.groupby("pair_row").joint_top1.mean().mean(), 2),
        "proxy_predicted_seen_pct": round(100 * details.predicted_seen.mean(), 2),
        "proxy_rank_blocked_pct": round(100 * details.rank_blocked.mean(), 2),
        "proxy_median_correct_rank": float(details.correct_pair_rank.median()),
    }


def summarize_proxy_pairs(details):
    grouped = details.groupby("pair_row", sort=True)
    accuracy = grouped[["joint_top1", "joint_top5", "proxy_only_top1", "proxy_only_top5",
                        "disease_top1", "disease_top5", "rank_blocked"]].mean() * 100
    return (accuracy.assign(images=grouped.size(),
                            median_correct_rank=grouped.correct_pair_rank.median()).reset_index()
            .merge(proxy_coverage, on="pair_row", validate="one_to_one"))


def ablation_deltas(checks):
    metrics = ("seen_top1", "proxy_top1", "proxy_top5", "harmonic_top1",
               "proxy_disease_head_top1", "proxy_disease_head_top5", "proxy_only_top1")
    rows = {row["recipe"]: row for row in checks}
    differences = []
    for method, control, component in (("A2b", "A2a", "decorrelation"),
                                       ("A2c", "A2b", "cross-crop SupCon"),
                                       ("A2c", "A2c_no_flip", "horizontal flip")):
        differences.append({"component": component, "method": method, "control": control,
                            "epoch": rows[method]["epoch"],
                            **{f"delta_{key}_pp": round(rows[method][key] - rows[control][key], 2)
                               for key in metrics}})
    return differences


def qualifies_for_calibration(candidate, raw):
    unseen_gain = candidate["proxy_top1"] - raw["proxy_top1"]
    seen_drop = raw["seen_top1"] - candidate["seen_top1"]
    return bool(unseen_gain >= MIN_PROXY_GAIN_PP - 1e-8
                and seen_drop <= MAX_SEEN_DROP_PP + 1e-8
                and candidate["harmonic_top1"] > raw["harmonic_top1"])


def select_gamma(tuning_scores):
    trials = [{"gamma": gamma, **dev_metrics(*tuning_scores, gamma=gamma)}
              for gamma in CALIBRATION_GRID]
    raw = trials[0]
    eligible = [row for row in trials[1:] if qualifies_for_calibration(row, raw)]
    proposed = max(eligible or trials[1:], key=lambda row: (
        row["harmonic_top1"], row["proxy_top1"], row["seen_top1"], -row["gamma"]))
    return proposed["gamma"], bool(eligible), trials


def development_run():
    if DEVELOPMENT_LOCK.is_file():
        saved = json.loads(DEVELOPMENT_LOCK.read_text(encoding="utf-8"))
        if saved["settings"] != settings:
            raise ValueError("Saved development settings differ. Archive the existing run folder before starting a new run.")
        print(f"Reusing {DEVELOPMENT_LOCK}")
        return saved
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    needs_flip = any(use_flip for _, _, use_flip in RECIPES.values())
    data = {name: cache_features(frame, name, with_flip=(name == "fit" and needs_flip))
            for name, frame in splits.items()}
    histories, best_epochs, trained = [], {}, {}
    tuning_data = [data[name] for name in ("tune_seen", "tune_proxy")]
    for recipe in RECIPES:
        heads, history, best, epoch_weights = fit_heads(data["fit"], recipe, "development", EPOCHS,
                                                        tune=tuning_data)
        histories.extend(history)
        trained[recipe] = (heads, epoch_weights)
        best_epochs[recipe] = best

    # Use the tuning winner's epoch for every recipe in the held-out raw ablation.
    winner = max(best_epochs, key=lambda name: selection_key(best_epochs[name]))
    best_epoch = best_epochs[winner]["epoch"]
    chosen_heads, chosen_weights = trained[winner]
    chosen_heads.load_state_dict(chosen_weights[best_epoch])
    tuning_scores = [score_cache(chosen_heads, group) for group in tuning_data]
    locked_gamma, passed_tune, calibration_grid = select_gamma(tuning_scores)

    checks, per_pair = [], []
    for recipe, (heads, epoch_weights) in trained.items():
        heads.load_state_dict(epoch_weights[best_epoch])
        check = [score_cache(heads, data[name]) for name in ("check_seen", "check_proxy")]
        details = proxy_details(check[1])
        metrics = dev_metrics(*check, details=details)
        checks.append({"recipe": recipe, "epoch": best_epoch, **metrics})
        per_pair.append(summarize_proxy_pairs(details).assign(recipe=recipe, epoch=best_epoch))
        if recipe == winner:
            raw_check = metrics
            proposed_check = dev_metrics(*check, gamma=locked_gamma)

    passed_check = qualifies_for_calibration(proposed_check, raw_check)
    calibration = {"locked_gamma": locked_gamma, "passed_tune": passed_tune,
                   "passed_development_check": passed_check}
    calibration_check = [
        {"scoring": "raw", "gamma": 0.0, **raw_check},
        {"scoring": "calibrated", "gamma": locked_gamma, **proposed_check},
    ]

    deltas = ablation_deltas(checks)
    pd.DataFrame(histories).to_csv(OUTPUT_DIR / "development_history.csv", index=False)
    pd.DataFrame(deltas).to_csv(OUTPUT_DIR / "raw_ablation_deltas.csv", index=False)
    pd.DataFrame(calibration_grid).to_csv(OUTPUT_DIR / "calibration_grid.csv", index=False)
    pd.DataFrame(calibration_check).to_csv(OUTPUT_DIR / "calibration_check.csv", index=False)
    pd.DataFrame(checks).to_csv(OUTPUT_DIR / "development_check.csv", index=False)
    pd.concat(per_pair, ignore_index=True).to_csv(OUTPUT_DIR / "development_per_pair.csv", index=False)
    proxy_coverage.to_csv(OUTPUT_DIR / "proxy_disease_coverage.csv", index=False)
    pd.concat([frame[["image", "crop_name", "disease_name", "pair_row"]].assign(split=name)
               for name, frame in splits.items()]).to_csv(OUTPUT_DIR / "split_manifest.csv", index=False)
    result = {"settings": settings, "backbone": backbone_record, "recipe": winner, "selected_epoch": best_epoch,
              "tune_best_raw": best_epochs, "check": checks, "ablation_deltas": deltas,
              "calibration": calibration, "calibration_check": calibration_check}
    atomic_json(DEVELOPMENT_LOCK, result)
    return result


development = development_run()
display(pd.DataFrame(development["check"]).round(2))
display(pd.DataFrame(development["ablation_deltas"]))
display(pd.DataFrame(development["calibration_check"])[
    ["scoring", "gamma", "seen_top1", "proxy_top1", "harmonic_top1"]])
print(f"Locked gamma: {development['calibration']['locked_gamma']:g}"
      f" | tune thresholds met: {development['calibration']['passed_tune']}"
      f" | check thresholds met: {development['calibration']['passed_development_check']}")
print(f"Selected {development['recipe']} at epoch {development['selected_epoch']}")
if development["tune_best_raw"][development["recipe"]]["harmonic_top1"] == 0:
    print("The selected raw checkpoint has zero tuning harmonic mean. Inspect the holdout diagnostics before interpreting generalization.")

Loading DINOv2 ViT-B/14 pretrained weights
fit: encoding batch 1/442 | 1.4s
fit: encoding batch 50/442 | 11.9s
fit: encoding batch 100/442 | 22.4s
fit: encoding batch 150/442 | 33.3s
fit: encoding batch 200/442 | 44.6s
fit: encoding batch 250/442 | 54.5s
fit: encoding batch 300/442 | 64.4s
fit: encoding batch 350/442 | 73.3s
fit: encoding batch 400/442 | 83.8s
fit: encoding batch 442/442 | 92.2s
tune_seen: encoding batch 1/24 | 0.4s
tune_seen: encoding batch 24/24 | 4.9s
tune_proxy: encoding batch 1/3 | 0.4s
tune_proxy: encoding batch 3/3 | 0.7s
check_seen: encoding batch 1/24 | 0.1s
check_seen: encoding batch 24/24 | 3.6s
check_proxy: encoding batch 1/9 | 0.1s
check_proxy: encoding batch 9/9 | 0.9s
development A2a epoch 1/20 | loss 5.259 | usable SupCon anchors 38.5% | tune seen 32.63% | proxy 2.70% | H 4.99%
development A2a epoch 2/20 | loss 2.764 | usable SupCon anchors 38.7% | tune seen 48.68% | proxy 0.00% | H 0.00%
development A2a epoch 3/20 | loss 2.084 | usable SupCon anchors 3

,recipe,epoch,seen_top1,proxy_top1,seen_top5,proxy_top5,harmonic_top1,proxy_crop_head_top1,proxy_disease_head_top1,proxy_disease_head_top5,proxy_only_top1,proxy_only_top5,proxy_macro_top1,proxy_predicted_seen_pct,proxy_rank_blocked_pct,proxy_median_correct_rank
0,A2a,20,77.89,6.06,96.32,60.61,11.25,90.15,7.58,49.24,83.33,100.0,7.81,93.94,77.27,3.0
1,A2b,20,77.89,6.06,96.32,61.36,11.25,90.15,7.58,49.24,83.33,100.0,7.81,93.94,77.27,3.0
2,A2c,20,78.16,6.06,96.32,60.61,11.25,90.15,7.58,50.76,82.58,100.0,7.81,93.94,76.52,3.0
3,A2c_no_flip,20,77.89,6.06,96.84,59.85,11.25,90.15,7.58,50.00,83.33,100.0,7.81,93.94,77.27,3.0


,component,method,control,epoch,delta_seen_top1_pp,delta_proxy_top1_pp,delta_proxy_top5_pp,delta_harmonic_top1_pp,delta_proxy_disease_head_top1_pp,delta_proxy_disease_head_top5_pp,delta_proxy_only_top1_pp
0,decorrelation,A2b,A2a,20,0.00,0.0,0.75,0.0,0.0,0.00,0.00
1,cross-crop SupCon,A2c,A2b,20,0.27,0.0,-0.75,0.0,0.0,1.52,-0.75
2,horizontal flip,A2c,A2c_no_flip,20,0.27,0.0,0.76,0.0,0.0,0.76,-0.75


,scoring,gamma,seen_top1,proxy_top1,harmonic_top1
0,raw,0.0,78.16,6.06,11.25
1,calibrated,6.0,75.53,69.70,72.50


Locked gamma: 6 | tune thresholds met: True | check thresholds met: True
Selected A2c at epoch 20


### Full-training refit

A full run refits the selected recipe on the complete train CSV for the selected number of epochs. It saves both scoring rules, the locked calibration value, the head checkpoint hash and the DINOv2 weight fingerprint. Rerunning a completed stage reuses its saved lock and files.

In [5]:
def refit_plan():
    return {
        "development_sha256": sha256_file(DEVELOPMENT_LOCK), "settings": settings,
        "recipe": development["recipe"],
        "gamma_by_scoring": {"raw": 0.0, "calibrated": development["calibration"]["locked_gamma"]},
        "calibration_development": development["calibration"],
        "seen_pairs": train_pairs.tolist(), "epochs": development["selected_epoch"],
        "backbone": development["backbone"],
    }


def final_training():
    if DEBUG:
        raise RuntimeError("Full refit requires DEBUG=False.")
    plan = refit_plan()
    if FINAL_LOCK.is_file():
        previous = json.loads(FINAL_LOCK.read_text(encoding="utf-8"))
        if (previous["plan"] != plan or not CHECKPOINT.is_file()
                or previous["checkpoint_sha256"] != sha256_file(CHECKPOINT)):
            raise RuntimeError("Final lock, checkpoint or settings conflict. Archive the existing run folder before starting a new run.")
        print(f"Reusing {CHECKPOINT}")
        return
    get_backbone()
    if backbone_record != plan["backbone"]:
        raise RuntimeError("Loaded backbone does not match the development fingerprint.")
    with_flip = RECIPES[plan["recipe"]][2]
    data = cache_features(train_full, "full train", with_flip=with_flip)
    heads, history, _, _ = fit_heads(data, plan["recipe"], "full_train", plan["epochs"])
    temporary = CHECKPOINT.with_suffix(".pt.tmp")
    torch.save({"weights": {name: tensor.cpu() for name, tensor in heads.state_dict().items()},
                "feature_width": data[0].shape[1]}, temporary)
    temporary.replace(CHECKPOINT)
    pd.DataFrame(history).to_csv(OUTPUT_DIR / "final_training_history.csv", index=False)
    atomic_json(FINAL_LOCK, {"plan": plan, "checkpoint_sha256": sha256_file(CHECKPOINT)})
    print(f"Saved frozen full-train checkpoint and rule to {OUTPUT_DIR}")


if DEBUG:
    print("DEBUG run finished after development.")
else:
    final_training()

full train: encoding batch 1/500 | 0.1s
full train: encoding batch 50/500 | 9.9s
full train: encoding batch 100/500 | 19.9s
full train: encoding batch 150/500 | 30.5s
full train: encoding batch 200/500 | 41.0s
full train: encoding batch 250/500 | 52.0s
full train: encoding batch 300/500 | 62.0s
full train: encoding batch 350/500 | 72.1s
full train: encoding batch 400/500 | 81.7s
full train: encoding batch 450/500 | 92.9s
full train: encoding batch 500/500 | 103.1s
full_train A2c epoch 1/20 | loss 5.537 | usable SupCon anchors 40.9%
full_train A2c epoch 2/20 | loss 3.025 | usable SupCon anchors 40.4%
full_train A2c epoch 3/20 | loss 2.342 | usable SupCon anchors 41.2%
full_train A2c epoch 4/20 | loss 1.970 | usable SupCon anchors 40.5%
full_train A2c epoch 5/20 | loss 1.725 | usable SupCon anchors 39.9%
full_train A2c epoch 6/20 | loss 1.561 | usable SupCon anchors 40.8%
full_train A2c epoch 7/20 | loss 1.418 | usable SupCon anchors 40.4%
full_train A2c epoch 8/20 | loss 1.312 | usable 

## 5. Evaluation

Official evaluation runs after the full-train checkpoint and both scoring rules are locked. The 10 official unseen pairs extend the candidate table to 333. Pair identities and image filenames are loaded before inference. Per-image ground-truth mappings are loaded after both official predictions are complete.

A single inference pass per split produces scores for both report rows. Each row uses the same 333 candidates and includes seen, unseen and overall Top-1/Top-5 and harmonic mean. Keep the raw result as the main assignment result pending confirmation for the calibrated comparison. The report also includes confusion matrices and brightness, blur and noise results. Prior experiments already evaluated these official test splits.

In [6]:
def official_vocabulary():
    # Project only concept columns. Do not load image-to-label mappings yet.
    vocabulary = pd.read_csv(team.TEST_UNSEEN_CSV, header=None, usecols=[1, 2, 3],
                             dtype=str, keep_default_na=False)
    vocabulary.columns = ["crop", "disease", "order"]
    if not vocabulary.empty and not vocabulary.iloc[0]["crop"].strip().isdigit():
        vocabulary = vocabulary.iloc[1:]
    vocabulary = vocabulary.astype("int64").drop_duplicates(["crop", "disease"]).sort_values("order")
    extra_pairs = torch.as_tensor(vocabulary[["crop", "disease"]].to_numpy(copy=True), dtype=torch.long)
    pairs = torch.cat((train_pairs, extra_pairs))
    if (len(extra_pairs) != team.NUM_UNSEEN_PAIRS or len(pairs.unique(dim=0)) != team.NUM_PAIRS
            or not set(extra_pairs[:, 0].tolist()) <= set(train_pairs[:, 0].tolist())
            or not set(extra_pairs[:, 1].tolist()) <= set(train_pairs[:, 1].tolist())):
        raise ValueError("Expected distinct official unseen pairs composed of training-derived concepts.")
    return pairs


def official_filenames(path):
    values = pd.read_csv(path, header=None, usecols=[0], dtype=str, keep_default_na=False).iloc[:, 0].tolist()
    header = {"image", "image_path", "image_name", "filename", "file_name", "file_path", "path"}
    if values and (values[0].strip().lower().replace(" ", "_") in header or
                   (len(values) > 1 and not Path(values[0]).suffix and Path(values[1]).suffix)):
        values = values[1:]
    names = [image_key(name) for name in values]
    if not names or len(set(names)) != len(names):
        raise ValueError(f"Expected unique image names in {path.name}.")
    return names


class OfficialImages(Dataset):
    def __init__(self, names):
        self.names = names

    def __len__(self):
        return len(self.names)

    def __getitem__(self, index):
        with Image.open(team.IMAGE_DIR / self.names[index]) as image:
            return image_transform(image.convert("RGB"))


@torch.inference_mode()
def predict_official(heads, pairs, names, title, perturb=None):
    loader = DataLoader(OfficialImages(names), batch_size=ENCODE_BATCH_SIZE, shuffle=False, num_workers=0)
    mean, std = (torch.tensor(value).view(1, 3, 1, 1) for value in team.NORM_STATS["imagenet"])
    batches = []
    for step, images in enumerate(loader, 1):
        if perturb is not None:
            pixels = (images * std + mean).clamp(0, 1)
            images = (perturb(pixels).clamp(0, 1) - mean) / std
        crop, disease, _, _ = heads(encode_images(images))
        scores = pair_scores(crop, disease, pairs).cpu()
        if scores.shape != (len(images), team.NUM_PAIRS) or not torch.isfinite(scores).all():
            raise ValueError("Final scores must be finite and rank all 333 candidates.")
        batches.append(scores)
        if step == 1 or step == len(loader) or step % 50 == 0:
            print(f"{title}: prediction batch {step}/{len(loader)}", flush=True)
    return torch.cat(batches)


def official_targets(csv_path, filenames, seen):
    frame = load_split(csv_path, final_evaluation=True)
    if frame.image.map(image_key).tolist() != filenames:
        raise ValueError("Official targets and predictions have different image orders.")
    labels = torch.as_tensor(frame.pair_row.to_numpy(copy=True), dtype=torch.long)
    if not ((labels < team.NUM_SEEN_PAIRS) == seen).all():
        raise ValueError("Official test split does not match seen membership.")
    return labels


def score_final(splits_and_scores, split_targets):
    return [
        {"split": split, "images": len(labels), "candidates": scores.shape[1],
         "top1": top_k_accuracy(scores, labels, 1), "top5": top_k_accuracy(scores, labels, 5)}
        for split, scores, labels in zip(("seen", "unseen", "overall"),
                                         splits_and_scores, split_targets)
    ]


def official_evaluation():
    if DEBUG:
        raise RuntimeError("Official evaluation requires DEBUG=False.")
    if not FINAL_LOCK.is_file() or not CHECKPOINT.is_file():
        raise FileNotFoundError("Complete full-train refit before enabling official evaluation.")
    frozen = json.loads(FINAL_LOCK.read_text(encoding="utf-8"))
    plan = frozen["plan"]
    if plan != refit_plan() or frozen["checkpoint_sha256"] != sha256_file(CHECKPOINT):
        raise RuntimeError("Current settings, run or weights do not match the frozen final plan.")
    eval_plan = {**frozen, "robustness": list(PERTURBATIONS)}
    if OFFICIAL_REPORT.is_file():
        report = json.loads(OFFICIAL_REPORT.read_text(encoding="utf-8"))
        if report["evaluation_plan"] != eval_plan:
            raise RuntimeError("Existing official results use a different frozen plan.")
        return report
    evaluation_lock = OUTPUT_DIR / "official_plan.json"
    if evaluation_lock.is_file() and json.loads(evaluation_lock.read_text(encoding="utf-8")) != eval_plan:
        raise RuntimeError("The previously frozen evaluation plan differs.")
    atomic_json(evaluation_lock, eval_plan)
    saved = torch.load(CHECKPOINT, map_location="cpu", weights_only=True)
    heads = ProjectedHeads(saved["feature_width"])
    heads.load_state_dict(saved["weights"], strict=True)
    heads = heads.to(DEVICE).eval()
    get_backbone()
    if backbone_record != plan["backbone"]:
        raise RuntimeError("Loaded backbone does not match the final fingerprint.")

    pairs = official_vocabulary()
    paths = (team.TEST_SEEN_CSV, team.TEST_UNSEEN_CSV)
    names = tuple(official_filenames(path) for path in paths)
    if (set(names[0]) & set(names[1]) or set(train_full.image) & (set(names[0]) | set(names[1]))):
        raise ValueError("Training and official split filenames overlap.")
    raw = tuple(predict_official(heads, pairs, group, title)
                for group, title in zip(names, ("seen", "unseen")))

    perturbed = {}
    with torch.random.fork_rng(devices=[]):
        torch.random.default_generator.manual_seed(team.SEED)
        for name, transform in PERTURBATIONS.items():
            perturbed[name] = predict_official(heads, pairs, names[0], name, transform)

    # Both scoring rules run before any per-image official targets are loaded.
    boundary = team.NUM_SEEN_PAIRS
    membership = torch.arange(len(pairs)) < boundary
    scored = {method: tuple(calibrate_scores(scores, membership, gamma) for scores in raw)
              for method, gamma in plan["gamma_by_scoring"].items()}
    scored_perturbations = {
        method: {name: calibrate_scores(scores, membership, gamma) for name, scores in perturbed.items()}
        for method, gamma in plan["gamma_by_scoring"].items()
    }
    if not torch.equal(pairs, get_pairs(final_evaluation=True)):
        raise ValueError("Official score columns differ from the shared pair order.")
    targets = (official_targets(paths[0], names[0], seen=True),
               official_targets(paths[1], names[1], seen=False))
    all_targets = (*targets, torch.cat(targets))

    summary, results_by_scoring = [], {}
    for method, scores in scored.items():
        gamma = plan["gamma_by_scoring"][method]
        results = score_final((*scores, torch.cat(scores)), all_targets)
        row = {"scoring": method, "gamma": gamma, "candidates": len(pairs)}
        row.update({f"{item['split']}_{metric}": item[metric]
                    for item in results for metric in ("top1", "top5")})
        row["harmonic_top1"] = harmonic_mean(row["seen_top1"], row["unseen_top1"])
        summary.append(row)
        robustness = {}
        for name, perturbed_scores in scored_perturbations[method].items():
            top1 = top_k_accuracy(perturbed_scores, targets[0], 1)
            robustness[name] = {"seen_top1": top1, "drop_pp": round(row["seen_top1"] - top1, 2)}
        results_by_scoring[method] = {
            "gamma": gamma, "results": results, "harmonic_mean": row["harmonic_top1"],
            "robustness": robustness,
            "seen_only_top1_diagnostic": top_k_accuracy(scores[0][:, :boundary], targets[0], 1),
            "unseen_only_top1_diagnostic": top_k_accuracy(scores[1][:, boundary:], targets[1] - boundary, 1),
        }
        confusion_name = (OUTPUT_DIR / f"official_seen_{method}").relative_to(team.RESULTS_DIR).as_posix()
        save_confusion(scores[0], targets[0], confusion_name, final_evaluation=True)

    metric_keys = ("seen_top1", "seen_top5", "unseen_top1", "unseen_top5",
                   "overall_top1", "overall_top5", "harmonic_top1")
    raw_row, calibrated_row = summary
    delta = {name: round(calibrated_row[name] - raw_row[name], 2) for name in metric_keys}
    report = {"evaluation_plan": eval_plan, "comparison": summary, "calibrated_minus_raw_pp": delta,
              "results_by_scoring": results_by_scoring}
    pd.DataFrame(summary).to_csv(OUTPUT_DIR / "official_comparison.csv", index=False)
    atomic_json(OFFICIAL_REPORT, report)
    return report


if DEBUG:
    print("Official evaluation skipped in DEBUG mode.")
else:
    official = official_evaluation()
    display(pd.DataFrame(official["comparison"]))
    calibration = official["evaluation_plan"]["plan"]["calibration_development"]
    if not (calibration["passed_tune"] and calibration["passed_development_check"]):
        print("Calibration missed the preset development targets. Interpret the calibrated row as exploratory.")

seen: prediction batch 1/125
seen: prediction batch 50/125
seen: prediction batch 100/125
seen: prediction batch 125/125
unseen: prediction batch 1/91
unseen: prediction batch 50/91
unseen: prediction batch 91/91
brightness: prediction batch 1/125
brightness: prediction batch 50/125
brightness: prediction batch 100/125
brightness: prediction batch 125/125
blur: prediction batch 1/125
blur: prediction batch 50/125
blur: prediction batch 100/125
blur: prediction batch 125/125
noise: prediction batch 1/125
noise: prediction batch 50/125
noise: prediction batch 100/125
noise: prediction batch 125/125


,scoring,gamma,candidates,seen_top1,seen_top5,unseen_top1,unseen_top5,overall_top1,overall_top5,harmonic_top1
0,raw,0.0,333,64.45,90.25,2.76,30.04,38.54,64.97,5.29
1,calibrated,6.0,333,59.20,88.70,54.70,94.68,57.31,91.21,56.86


## 6. Save

Open `raw_ablation_deltas.csv` for the component-by-component changes in proxy-unseen Top-1, disease recognition and harmonic mean. `development_per_pair.csv` and `proxy_disease_coverage.csv` show which combinations improve and how much fit support each disease had. Use `official_comparison.csv` to compare raw and calibrated results from the same checkpoint. The calibration grid and check CSV record how the penalty was locked. A full run exports the tuning curves, official comparison figure and a text summary with checkpoint fingerprints for the report.

In [7]:
def save_research_figures():
    from matplotlib import pyplot as plt

    history = pd.read_csv(OUTPUT_DIR / "development_history.csv")
    selected_epoch = development["selected_epoch"]
    development_metrics = (
        ("tune_seen_top1", "Seen pair Top-1"),
        ("tune_proxy_top1", "Proxy-unseen pair Top-1"),
        ("tune_harmonic_top1", "Pair harmonic mean"),
        ("tune_proxy_disease_head_top1", "Proxy-unseen disease head Top-1"),
    )
    fig, axes = plt.subplots(2, 2, figsize=(9, 6), sharex=True, layout="constrained")
    for ax, (metric, label) in zip(axes.flat, development_metrics):
        for recipe, rows in history.groupby("recipe", sort=False):
            ax.plot(rows["epoch"], rows[metric], marker=".", label=recipe)
        ax.axvline(selected_epoch, color="#6b7280", ls="--", lw=1)
        ax.set_title(label)
        ax.set_ylabel("Percent")
        ax.grid(alpha=0.2)
    axes[0, 0].legend(loc="best", fontsize="small")
    for ax in axes[-1]:
        ax.set_xlabel("Epoch")
    fig.suptitle("Training-derived tuning results · raw scores")
    fig.savefig(OUTPUT_DIR / "development_curves.png", dpi=180)
    plt.close(fig)

    metrics = ["seen_top1", "unseen_top1", "overall_top1", "harmonic_top1"]
    rows = official["comparison"]
    colors = ("#246189", "#af6229")
    x = np.arange(len(metrics))
    fig, ax = plt.subplots(figsize=(8, 4.2), layout="constrained")
    for i, row in enumerate(rows):
        values = [row[k] for k in metrics]
        bars = ax.bar(x + (i - 0.5) * 0.35, values, width=0.35, color=colors[i],
                      label=f"{row['scoring'].title()} (gamma={row['gamma']:g})")
        ax.bar_label(bars, fmt="%.2f", padding=3, fontsize=8)
    ax.set_xticks(x, ["Seen", "Unseen", "Overall", "Harmonic mean"])
    ax.set_ylabel("Top-1 (%)")
    upper = max(row[key] for row in rows for key in metrics)
    ax.set_ylim(0, max(10, upper * 1.23))
    ax.set_title("Official evaluation · same checkpoint and 333 candidates")
    ax.legend()
    ax.set_axisbelow(True)
    ax.grid(axis="y", alpha=0.2)
    fig.savefig(OUTPUT_DIR / "official_comparison.png", dpi=180)
    plt.close(fig)


def save_run_summary():
    plan = official["evaluation_plan"]["plan"]
    metrics = pd.DataFrame(official["comparison"])
    check = pd.DataFrame(development["check"])[
        ["recipe", "epoch", "seen_top1", "proxy_top1", "harmonic_top1",
         "proxy_disease_head_top1", "proxy_disease_head_top5"]]
    notes = [
        "COS30082 Approach 2 — final run evidence",
        f"Recipe {plan['recipe']} | refit epochs {plan['epochs']} | seed {settings['seed']}",
        f"DINOv2 weight SHA-256 {plan['backbone']['state_sha256']}",
        f"Head checkpoint SHA-256 {official['evaluation_plan']['checkpoint_sha256']}",
        "",
        "Held-out development check · matched epoch · raw scores",
        check.to_string(index=False),
        "",
        "Official result · 333 candidates · same checkpoint and images for both scoring rules",
        metrics.to_string(index=False),
        "",
        "Raw is the main comparison. Calibrated is exploratory pending assignment confirmation.",
        "The development and official metrics come from different splits.",
        "See official_report.json for per-method corruption results and calibrated-minus-raw differences.",
    ]
    path = OUTPUT_DIR / "run_summary.txt"
    temporary = path.with_suffix(".tmp")
    temporary.write_text("\n".join(notes) + "\n", encoding="utf-8")
    temporary.replace(path)


if not DEBUG:
    save_research_figures()
    save_run_summary()
print(f"Run folder: {OUTPUT_DIR.resolve()}")
for name in ("development_history.csv", "development_check.csv", "development_per_pair.csv",
             "raw_ablation_deltas.csv", "proxy_disease_coverage.csv", "split_manifest.csv",
             "calibration_grid.csv", "calibration_check.csv", "final_heads.pt", "final_lock.json",
             "official_comparison.csv", "official_report.json", "official_seen_raw_confusion.png",
             "official_seen_calibrated_confusion.png", "development_curves.png",
             "official_comparison.png", "run_summary.txt"):
    if (OUTPUT_DIR / name).is_file():
        print(f"  {name}")

Run folder: C:\Users\104394862\Downloads\COS30082-Plant-Disease-OOD\results\approach2_final_dinov2
  development_history.csv
  development_check.csv
  development_per_pair.csv
  raw_ablation_deltas.csv
  proxy_disease_coverage.csv
  split_manifest.csv
  calibration_grid.csv
  calibration_check.csv
  final_heads.pt
  final_lock.json
  official_comparison.csv
  official_report.json
  official_seen_raw_confusion.png
  official_seen_calibrated_confusion.png
  development_curves.png
  official_comparison.png
  run_summary.txt


## Research references

- Oquab et al. (2024), *DINOv2: Learning Robust Visual Features without Supervision*. Frozen feature extractor. [Paper](https://arxiv.org/abs/2304.07193)
- Khosla et al. (2020), *Supervised Contrastive Learning*. This project restricts disease positives to examples on different crops. [Paper](https://arxiv.org/abs/2004.11362)
- Chao et al. (2016), *An Empirical Study and Analysis of Generalized Zero-Shot Learning for Object Recognition in the Wild*. Calibrated stacking adapted here to pair membership. [Paper](https://arxiv.org/abs/1605.04253)
- Hendrycks and Dietterich (2019), *Benchmarking Neural Network Robustness to Common Corruptions and Perturbations*. The project's three fixed perturbations are a synthetic stress test. [Paper](https://arxiv.org/abs/1903.12261)